# Running StormCast Ensemble Inference

Ensemble StormCast inference workflow.

This example will demonstrate how to run a simple inference workflow to generate a
ensemble forecast using StormCast. For details about the stormcast model,
see

 - https://arxiv.org/abs/2408.10958

## Set Up
All workflows inside Earth2Studio require constructed components to be
handed to them. In this example, we run an ensemble forecast with
[`earth2studio.run.Pipeline`][earth2studio.run.Pipeline], where each ensemble member
is a work item produced by [`earth2studio.run.work_items`][earth2studio.run.work_items].

We need the following:

- Prognostic Model: Use the built in StormCast-CONUS Model [`earth2studio.models.px.StormCastCONUS`][earth2studio.models.px.StormCastCONUS].
  We will not perturb the initial data because StormCast has stochastic generation of ensemble members.
- Datasource: Pull data from the HRRR data api [`earth2studio.data.HRRR`][earth2studio.data.HRRR].
- IO Backend: Let's save the outputs into a Zarr store [`earth2studio.io.ZarrBackend`][earth2studio.io.ZarrBackend].

StormCast-CONUS also requires global conditioning. Its default sources supply
GFS_FX [`earth2studio.data.GFS_FX`][earth2studio.data.GFS_FX] forecasts regridded
onto the HRRR grid.

In [ ]:
import numpy as np
from loguru import logger
from tqdm import tqdm

logger.remove()
logger.add(lambda msg: tqdm.write(msg, end=""), colorize=True)

import os

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()  # TODO: make common example prep function

from earth2studio.data import HRRR
from earth2studio.io import ZarrBackend
from earth2studio.models.px import StormCastCONUS

# Load the default model package which downloads the checkpoint from HuggingFace.
# Global conditioning comes from the model's default sources: GFS forecasts
# regridded onto the HRRR grid.
package = StormCastCONUS.load_default_package()
model = StormCastCONUS.load_model(package)

# Create the data source
data = HRRR()

# Create the IO handler, store in memory
io = ZarrBackend()


## Execute the Workflow
With all components initialized, the pipeline fetches the initial condition and
conditioning, rolls out each ensemble member and writes it into the IO object, which
can then be used to post process. Some have additional APIs that can be handy for post-processing or
saving to file. Check the API docs for more information.

For the forecast we will predict for 4 hours

In [ ]:
import torch

from earth2studio.run import Pipeline, work_items

nsteps = 4
nensemble = 2

date = "2022-11-04T18:00:00"
items = work_items(
    [np.datetime64(date)],
    final_lead_time=np.timedelta64(nsteps, "h"),
    members=nensemble,
)
Pipeline.from_model(
    model,
    data,
    output={"forecast": io},
    output_variables={"forecast": ["t2m", "refc"]},
    device="cuda" if torch.cuda.is_available() else "cpu",
).run(items)

print(io.root.tree())


## Post Processing
The last step is to post process our results. Cartopy is a great library for plotting
fields on projections of a sphere. Start with plotting the reflectivity.

Notice that the Zarr IO function has additional APIs to interact with the stored data.

In [ ]:
import cartopy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

forecast = f"{date}"
step = nsteps  # 4 hours, since lead_time = 1 hr

# Create a correct Lambert Conformal projection
projection = ccrs.LambertConformal(
    central_longitude=262.5,
    central_latitude=38.5,
    standard_parallels=(38.5, 38.5),
    globe=ccrs.Globe(semimajor_axis=6371229, semiminor_axis=6371229),
)


# Get the lat lon arrays from the model
def plot_(axi, data, title, cmap, vmin=None, vmax=None):
    """Convenience function for plotting pcolormesh."""
    # Plot the field using pcolormesh
    im = axi.pcolormesh(
        model.lon,
        model.lat,
        data,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    plt.colorbar(im, ax=axi, shrink=0.6, pad=0.04)
    # Set title
    axi.set_title(title)

    # Add coastlines and gridlines
    axi.coastlines()
    axi.gridlines()

    # Set state lines
    axi.add_feature(
        cartopy.feature.STATES.with_scale("50m"),
        linewidth=0.5,
        edgecolor="black",
        zorder=2,
    )


# Plot refc
variable = "refc"
cmap = "gist_ncar"
x = io[variable]

plt.close("all")
fig, (ax1, ax2, ax3) = plt.subplots(
    nrows=1, ncols=3, subplot_kw={"projection": projection}, figsize=(20, 6)
)
plot_(
    ax1,
    np.where(x[0, 0, step] > 0, x[0, 0, step], np.nan),
    f"{forecast} - Lead time: {step}hrs - Member: {0}",
    cmap,
    vmin=0,
    vmax=60,
)
plot_(
    ax2,
    np.where(x[1, 0, step] > 0, x[1, 0, step], np.nan),
    f"{forecast} - Lead time: {step}hrs - Member: {1}",
    cmap,
    vmin=0,
    vmax=60,
)
plot_(
    ax3,
    np.where(x[:, 0, step].mean(axis=0) > 0, x[:, 0, step].std(axis=0), np.nan),
    f"{forecast} - Lead time: {step}hrs - Std",
    cmap,
    vmin=0,
    vmax=60,
)
plt.savefig(f"outputs/10_{date}_{variable}_{step}_ensemble.jpg")


Lets also plot the surface temperature field.

In [ ]:
# Plot 2-meter temperature
variable = "t2m"
cmap = "Spectral_r"
x = io[variable]

plt.close("all")
fig, (ax1, ax2, ax3) = plt.subplots(
    nrows=1, ncols=3, subplot_kw={"projection": projection}, figsize=(20, 6)
)
plot_(
    ax1,
    x[0, 0, step],
    f"{forecast} - Lead time: {step}hrs - Member: {0}",
    cmap,
)
plot_(
    ax2,
    io[variable][1, 0, step],
    f"{forecast} - Lead time: {step}hrs - Member: {1}",
    cmap,
)
plot_(
    ax3,
    x[:, 0, step].std(axis=0),
    f"{forecast} - Lead time: {step}hrs - Std",
    cmap,
)
plt.savefig(f"outputs/10_{date}_{variable}_{step}_ensemble.jpg")
